# Regression and Multi-Output Models

**Chapter 7: Predicting Soccer Outcomes with Deep Learning**

## Learning Objectives

- Understand regression with neural networks
- Learn about Poisson regression for count data (goals)
- Build models to predict goal counts
- Understand multi-task learning (MTL)
- Build multi-output models that predict both home and away goals
- Compare single-task vs multi-task approaches

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")

## Introduction

In previous notebooks, we predicted **categories** (win/draw/loss). Now we'll predict **counts** - specifically, the number of goals scored.

This is fundamentally different:
- **Classification**: Discrete categories (win, draw, loss)
- **Regression**: Continuous or count values (0, 1, 2, 3... goals)

For goals, we'll use **Poisson regression**, which assumes goals follow a Poisson distribution - perfect for modeling rare events like goals in soccer.

## Generate Simulated Soccer Data with Goals

In [ ]:
# Generate simulated match data with goals
np.random.seed(42)
n_matches = 1000

# Features
shots_home = np.random.poisson(12, n_matches)
possession_home = np.random.normal(50, 15, n_matches)
xg_home = np.random.gamma(2, 0.6, n_matches)
corners_home = np.random.poisson(5, n_matches)
fouls_home = np.random.poisson(10, n_matches)

shots_away = np.random.poisson(10, n_matches)
possession_away = 100 - possession_home
xg_away = np.random.gamma(1.5, 0.6, n_matches)
corners_away = np.random.poisson(4, n_matches)
fouls_away = np.random.poisson(11, n_matches)

# Create DataFrame
df = pd.DataFrame({
    'shots_home': shots_home,
    'possession_home': possession_home,
    'xg_home': xg_home,
    'corners_home': corners_home,
    'fouls_home': fouls_home,
    'shots_away': shots_away,
    'possession_away': possession_away,
    'xg_away': xg_away,
    'corners_away': corners_away,
    'fouls_away': fouls_away
})

# Generate goals based on xG (with Poisson distribution)
df['FTHG'] = np.random.poisson(df['xg_home'])  # Full-Time Home Goals
df['FTAG'] = np.random.poisson(df['xg_away'])  # Full-Time Away Goals

print(f"Dataset shape: {df.shape}")
print(f"\nGoal statistics:")
print(df[['FTHG', 'FTAG']].describe())
print(f"\nAverage home goals: {df['FTHG'].mean():.2f}")
print(f"Average away goals: {df['FTAG'].mean():.2f}")

In [ ]:
# Visualize goal distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Home goals
axes[0].hist(df['FTHG'], bins=range(0, df['FTHG'].max()+2), 
             alpha=0.7, edgecolor='black', color='blue')
axes[0].set_xlabel('Goals', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Home Goals', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Away goals
axes[1].hist(df['FTAG'], bins=range(0, df['FTAG'].max()+2), 
             alpha=0.7, edgecolor='black', color='red')
axes[1].set_xlabel('Goals', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Away Goals', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Goals follow a Poisson-like distribution - most matches have 0-3 goals")

---

# Task 3: Predicting Home Team Goals

We've predicted categorical outcomes (win/draw/loss). Now let's predict **how many goals** are scored.

Because goals are **counts** (non-negative integers), we use **Poisson regression**. It assumes the number of goals follows a Poisson distribution, which works well for relatively rare events like goals in soccer.

## Understanding Poisson Distribution

The Poisson distribution models the probability of a given number of events occurring in a fixed interval.

For goals, it's characterized by one parameter: **λ (lambda)** - the expected number of goals.

If λ = 1.7, the probabilities are:
- P(0 goals) ≈ 18%
- P(1 goal) ≈ 31%
- P(2 goals) ≈ 27%
- P(3 goals) ≈ 15%
- etc.

In [ ]:
# Visualize Poisson distribution
lambda_val = 1.7
x = np.arange(0, 8)
poisson_probs = stats.poisson.pmf(x, lambda_val)

plt.figure(figsize=(10, 5))
plt.bar(x, poisson_probs, alpha=0.7, edgecolor='black')
plt.xlabel('Number of Goals', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title(f'Poisson Distribution (λ = {lambda_val})', fontsize=14, fontweight='bold')
plt.xticks(x)
plt.grid(True, alpha=0.3, axis='y')

for i, prob in enumerate(poisson_probs):
    plt.text(i, prob + 0.01, f'{prob:.2%}', ha='center', fontsize=10)

plt.show()

print(f"Expected value (mean): {lambda_val}")
print(f"Variance: {lambda_val} (same as mean in Poisson distribution)")

## Data Preparation for Regression

In [ ]:
# Select features and target
feature_cols = ['shots_home', 'possession_home', 'xg_home', 'corners_home', 'fouls_home',
                'shots_away', 'possession_away', 'xg_away', 'corners_away', 'fouls_away']

X = df[feature_cols].values
y_home_goals = df['FTHG'].values
y_away_goals = df['FTAG'].values

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_home_tensor = torch.tensor(y_home_goals, dtype=torch.float32)
y_away_tensor = torch.tensor(y_away_goals, dtype=torch.float32)

# Train-test split
X_train, X_test, y_home_train, y_home_test = train_test_split(
    X_tensor, y_home_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders
train_data = TensorDataset(X_train, y_home_train)
test_data = TensorDataset(X_test, y_home_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")
print(f"\nTarget statistics (home goals):")
print(f"  Mean: {y_home_train.mean():.2f}")
print(f"  Std: {y_home_train.std():.2f}")
print(f"  Min: {y_home_train.min():.0f}")
print(f"  Max: {y_home_train.max():.0f}")

## Poisson Regression Model

For count data, we need to ensure the network's output is always **positive** (negative goals make no sense!).

We do this by adding a **Softplus** activation at the end. This guarantees outputs > 0.

The output represents **λ** (lambda) - the expected number of goals.

In [ ]:
class PoissonRegression(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16, 1)
        self.softplus = nn.Softplus()  # ensures positive outputs
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return self.softplus(x)  # λ parameter (expected goals)

# Create model
input_size = X_train.shape[1]
model_poisson = PoissonRegression(input_size)

# Poisson Negative Log-Likelihood Loss
criterion_poisson = nn.PoissonNLLLoss(log_input=False)
optimizer_poisson = optim.Adam(model_poisson.parameters(), lr=0.001)

print(f"Model architecture:")
print(model_poisson)
print(f"\nTotal parameters: {sum(p.numel() for p in model_poisson.parameters())}")

### Understanding the Architecture

- **Hidden layers with ReLU**: Learn complex combinations (e.g., "high possession + strong attack → more goals")
- **Softplus output**: Outputs positive λ (expected goals)

If λ = 1.8, we interpret that as "on average, the home team scores about 1.8 goals in matches like this."

## Training with Early Stopping

We'll implement **early stopping** - halt training if validation performance stops improving.

This is like a coach deciding: "We've trained enough this week - more practice won't make us sharper and might even wear us out."

In [ ]:
# Training loop with early stopping
num_epochs = 200
best_val_loss = float('inf')
patience = 20
patience_counter = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training phase
    model_poisson.train()
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        optimizer_poisson.zero_grad()
        outputs = model_poisson(inputs).squeeze()
        loss = criterion_poisson(outputs, targets)
        loss.backward()
        optimizer_poisson.step()
        running_loss += loss.item()
    
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model_poisson.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model_poisson(inputs).squeeze()
            val_loss += criterion_poisson(outputs, targets).item()
    
    val_loss = val_loss / len(test_loader)
    val_losses.append(val_loss)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        best_model_state = model_poisson.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            # Restore best model
            model_poisson.load_state_dict(best_model_state)
            break
    
    # Logging
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print(f"\nTraining complete! Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Progress with Early Stopping', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## Evaluating the Poisson Model

In [ ]:
# Make predictions
model_poisson.eval()
with torch.no_grad():
    y_pred_lambda = model_poisson(X_test).squeeze().numpy()

y_true = y_home_test.numpy()

# Calculate metrics
mse = mean_squared_error(y_true, y_pred_lambda)
mae = mean_absolute_error(y_true, y_pred_lambda)
rmse = np.sqrt(mse)

print(f"Regression Metrics:")
print(f"  Mean Squared Error (MSE): {mse:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"  Mean Absolute Error (MAE): {mae:.4f}")
print(f"\nInterpretation:")
print(f"  On average, predictions are off by {mae:.2f} goals")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(10, 5))
plt.scatter(y_true, y_pred_lambda, alpha=0.5)
plt.plot([0, y_true.max()], [0, y_true.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Goals', fontsize=12)
plt.ylabel('Predicted λ (Expected Goals)', fontsize=12)
plt.title('Predicted vs Actual Home Goals', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

### Interpreting Poisson Predictions

The model outputs **λ** (expected goals). We can use this to calculate probabilities of exact outcomes!

In [ ]:
# Example: Calculate probabilities for a specific match
example_lambda = y_pred_lambda[0]
print(f"Predicted λ: {example_lambda:.2f}")
print(f"\nProbabilities of scoring exactly:")

for goals in range(6):
    prob = stats.poisson.pmf(goals, example_lambda)
    print(f"  {goals} goals: {prob:.2%}")

print(f"\nActual goals scored: {int(y_true[0])}")

In [ ]:
# Show sample predictions
print("Sample Predictions:")
print("\nActual | Predicted λ | Most Likely")
print("-" * 45)
for i in range(15):
    actual = int(y_true[i])
    pred_lambda = y_pred_lambda[i]
    most_likely = int(np.round(pred_lambda))
    print(f"{actual:^6} | {pred_lambda:^11.2f} | {most_likely:^11}")

---

# Task 4: Multi-Task Learning - Predicting Both Teams' Goals

So far, we've only predicted home goals. But real soccer analysis requires predicting **both teams' goals** together.

We could train two separate models, but that misses an opportunity: **the same features influence both outcomes**.

**Multi-Task Learning (MTL)** trains a single network with:
- **Shared layers**: Capture general soccer knowledge
- **Task-specific heads**: Specialize in home or away goals

This makes the model:
- ✅ **More efficient**: One model instead of two
- ✅ **More accurate**: Leverages shared patterns
- ✅ **More robust**: Sharing helps avoid overfitting

Think of it like training a squad together - all players benefit from the same fitness drills (shared layers), but strikers and defenders still practice specialized roles (task-specific heads).

## Multi-Task Model Architecture

In [ ]:
class MultiTaskMLP(nn.Module):
    def __init__(self, input_size, hidden_size=64, shared_size=32):
        super().__init__()
        
        # Shared layers (learn general soccer patterns)
        self.shared_layer1 = nn.Linear(input_size, hidden_size)
        self.shared_relu = nn.ReLU()
        self.shared_layer2 = nn.Linear(hidden_size, shared_size)
        
        # Home goals head
        self.home_goals_head = nn.Sequential(
            nn.Linear(shared_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Softplus()  # ensure non-negative goals
        )
        
        # Away goals head
        self.away_goals_head = nn.Sequential(
            nn.Linear(shared_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Softplus()  # ensure non-negative goals
        )
    
    def forward(self, x):
        # Shared feature extraction
        x = self.shared_layer1(x)
        x = self.shared_relu(x)
        shared_features = self.shared_layer2(x)
        
        # Task-specific predictions
        home_goals = self.home_goals_head(shared_features)
        away_goals = self.away_goals_head(shared_features)
        
        return home_goals, away_goals

# Create model
model_mtl = MultiTaskMLP(input_size, hidden_size=64, shared_size=32)

print(f"Multi-Task Model Architecture:")
print(model_mtl)
print(f"\nTotal parameters: {sum(p.numel() for p in model_mtl.parameters())}")

## Prepare Data for Multi-Task Learning

In [ ]:
# We need both home and away goals as targets
X_train_mtl, X_test_mtl, y_home_train_mtl, y_home_test_mtl, y_away_train_mtl, y_away_test_mtl = train_test_split(
    X_tensor, y_home_tensor, y_away_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders with both targets
train_data_mtl = TensorDataset(X_train_mtl, y_home_train_mtl, y_away_train_mtl)
test_data_mtl = TensorDataset(X_test_mtl, y_home_test_mtl, y_away_test_mtl)

train_loader_mtl = DataLoader(train_data_mtl, batch_size=32, shuffle=True)
test_loader_mtl = DataLoader(test_data_mtl, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train_mtl)} matches")
print(f"Test set: {len(X_test_mtl)} matches")

## Training the Multi-Task Model

We calculate loss for **both tasks** and combine them:

In [ ]:
# Loss and optimizer
criterion_mtl = nn.PoissonNLLLoss(log_input=False)
optimizer_mtl = optim.Adam(model_mtl.parameters(), lr=0.001)

# Training loop
num_epochs = 150
train_losses_mtl = []

for epoch in range(num_epochs):
    model_mtl.train()
    running_loss = 0.0
    
    for inputs, home_targets, away_targets in train_loader_mtl:
        optimizer_mtl.zero_grad()
        
        # Forward pass - get both predictions
        home_pred, away_pred = model_mtl(inputs)
        home_pred = home_pred.squeeze()
        away_pred = away_pred.squeeze()
        
        # Calculate loss for both tasks
        loss_home = criterion_mtl(home_pred, home_targets)
        loss_away = criterion_mtl(away_pred, away_targets)
        
        # Combined loss (equal weighting)
        total_loss = loss_home + loss_away
        
        # Backward pass
        total_loss.backward()
        optimizer_mtl.step()
        
        running_loss += total_loss.item()
    
    avg_loss = running_loss / len(train_loader_mtl)
    train_losses_mtl.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Total Loss: {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Visualize training
plt.figure(figsize=(10, 5))
plt.plot(train_losses_mtl, linewidth=2, color='purple')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Combined Loss', fontsize=12)
plt.title('Multi-Task Training Progress', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

## Evaluating the Multi-Task Model

In [ ]:
# Make predictions
model_mtl.eval()
with torch.no_grad():
    home_preds_mtl, away_preds_mtl = model_mtl(X_test_mtl)
    home_preds_mtl = home_preds_mtl.squeeze().numpy()
    away_preds_mtl = away_preds_mtl.squeeze().numpy()

y_home_true = y_home_test_mtl.numpy()
y_away_true = y_away_test_mtl.numpy()

# Calculate metrics for both tasks
print("Multi-Task Model Performance:")
print("\nHome Goals:")
print(f"  MAE: {mean_absolute_error(y_home_true, home_preds_mtl):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_home_true, home_preds_mtl)):.4f}")

print("\nAway Goals:")
print(f"  MAE: {mean_absolute_error(y_away_true, away_preds_mtl):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_away_true, away_preds_mtl)):.4f}")

In [ ]:
# Visualize predictions for both tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Home goals
axes[0].scatter(y_home_true, home_preds_mtl, alpha=0.5, color='blue')
axes[0].plot([0, y_home_true.max()], [0, y_home_true.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Home Goals', fontsize=12)
axes[0].set_ylabel('Predicted λ (Home)', fontsize=12)
axes[0].set_title('Home Goals Prediction', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Away goals
axes[1].scatter(y_away_true, away_preds_mtl, alpha=0.5, color='red')
axes[1].plot([0, y_away_true.max()], [0, y_away_true.max()], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Away Goals', fontsize=12)
axes[1].set_ylabel('Predicted λ (Away)', fontsize=12)
axes[1].set_title('Away Goals Prediction', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Show sample predictions for full matches
print("Sample Match Predictions:")
print("\nActual Score | Predicted Score (λ) | Most Likely Score")
print("-" * 60)
for i in range(15):
    actual_home = int(y_home_true[i])
    actual_away = int(y_away_true[i])
    pred_home = home_preds_mtl[i]
    pred_away = away_preds_mtl[i]
    likely_home = int(np.round(pred_home))
    likely_away = int(np.round(pred_away))
    
    print(f"{actual_home:^5}-{actual_away:<5} | {pred_home:^7.2f}-{pred_away:<7.2f} | {likely_home:^7}-{likely_away:<7}")

## Summary

In this notebook, you learned:

✅ **Regression with neural networks** for predicting continuous values

✅ **Poisson regression** for modeling count data (goals)

✅ How to use **Softplus activation** to ensure positive outputs

✅ **Early stopping** to prevent overfitting

✅ **Multi-task learning (MTL)** to predict multiple related outcomes

✅ How to build models with **shared layers** and **task-specific heads**

✅ The advantages of MTL: efficiency, accuracy, and robustness

### Key Takeaways

- **Poisson regression** is ideal for rare events like goals
- **λ (lambda)** represents expected goals and can be used to calculate probabilities
- **Multi-task learning** leverages shared patterns across related tasks
- Predicting both teams' goals together is more effective than separate models

## Practice Exercises

1. **Compare single-task vs multi-task**: Train separate models for home and away goals, then compare performance to the MTL model.

2. **Weighted loss**: Experiment with different weights for home vs away loss (e.g., `0.6 * loss_home + 0.4 * loss_away`).

3. **Deeper shared layers**: Add more shared layers to the MTL model. Does it improve performance?

4. **Match outcome from goals**: Use the predicted goals to calculate win/draw/loss probabilities.

5. **Total goals**: Modify the model to predict total goals (home + away) as a third task.

6. **Confidence intervals**: Use the Poisson distribution to calculate confidence intervals for goal predictions.

7. **Feature analysis**: Which features are most important for predicting goals? Try removing features one at a time.